## Exercise 1: Convolutional Autoencoder
**Dataset Used:** Fashion-MNIST (keras.datasets)

1. Build Conv-Autoencoder: Encoder Conv2D(32)+MaxPool, Conv2D(16)+MaxPool; Decoder mirrored.
2. Compare MSE of Conv vs Dense on 100 test images.
3. t-SNE visualization of Bottleneck Features.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Dense, Flatten, Reshape
from tensorflow.keras.models import Model
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Conv Autoencoder
input_img = Input(shape=(28, 28, 1))
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(16, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same', name='bottleneck')(x)

x = Conv2D(16, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

autoencoder_conv = Model(input_img, decoded)
autoencoder_conv.compile(optimizer='adam', loss='binary_crossentropy')

# Train 20 Epochs
autoencoder_conv.fit(x_train, x_train, epochs=20, batch_size=256, shuffle=True, validation_data=(x_test, x_test), verbose=1)

# Dense Autoencoder for comparison
x_train_flat = x_train.reshape((len(x_train), 784))
x_test_flat = x_test.reshape((len(x_test), 784))
input_flat = Input(shape=(784,))
d_enc = Dense(128, activation='relu')(input_flat)
d_enc = Dense(32, activation='relu')(d_enc)
d_dec = Dense(128, activation='relu')(d_enc)
d_dec = Dense(784, activation='sigmoid')(d_dec)
autoencoder_dense = Model(input_flat, d_dec)
autoencoder_dense.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder_dense.fit(x_train_flat, x_train_flat, epochs=20, batch_size=256, shuffle=True, validation_data=(x_test_flat, x_test_flat), verbose=0)

# Compare MSE on 100 test images
pred_conv = autoencoder_conv.predict(x_test[:100])
pred_dense = autoencoder_dense.predict(x_test_flat[:100]).reshape((100, 28, 28, 1))
mse_conv = np.mean((x_test[:100] - pred_conv)**2)
mse_dense = np.mean((x_test[:100] - pred_dense)**2)
print(f"MSE Conv: {mse_conv:.4f}, MSE Dense: {mse_dense:.4f}")

# t-SNE on bottleneck
encoder = Model(input_img, encoded)
features = encoder.predict(x_test[:1000])
features_flat = features.reshape((1000, -1))
tsne_res = TSNE(n_components=2, random_state=42).fit_transform(features_flat)
plt.scatter(tsne_res[:,0], tsne_res[:,1], c=y_test[:1000], cmap='tab10')
plt.colorbar()
plt.title('t-SNE of Bottleneck Features')
plt.show()
